In [1]:
# Part 1 - Imports

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

import torch

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23852\325804310.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# Part 2 - Load Embeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23852\1810976030.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [3]:
# Part 3 - Load FAISS

vectorstore = FAISS.load_local(
    "finance_faiss",
    embeddings,
    allow_dangerous_deserialization=True
)

In [4]:
# Part 4 - Load Gemma

generator = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it",
    device_map="auto",
    torch_dtype=torch.float16
)

print("Gemma Loaded Successfully")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


Gemma Loaded Successfully


In [5]:
# Part 5 - Ask Question

query = "What were Infosys revenues?"

In [6]:
# Part 6 - Retrieve Documents

docs_with_scores = vectorstore.similarity_search_with_score(
    query,
    k=3
)

print("\nTOP RETRIEVED DOCUMENTS\n")

for doc, score in docs_with_scores:

    print(
        f"Score: {score:.4f}"
    )

    print(
        doc.metadata
    )

    print(
        doc.page_content[:500]
    )

    print("=" * 80)


TOP RETRIEVED DOCUMENTS

Score: 0.6777
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.3 (Macintosh)', 'creationdate': '2025-06-01T23:16:08+05:30', 'author': 'Infosys Limited', 'keywords': 'Infosys Integrated Annual Report 2024-25; Infosys; Integrated Annual Report; Infosys Integrated Annual Report; 2024-25; 2024; 2025; Letter to the shareholder; AI; AI your Enterprise; Generative AI; Interactive; Navigate your next; Our Purpose; amplify; human potential; create; next opportunity; people; businesses; communities; C-LIFE; Nandan M. Nilekani; Salil Parekh; CEO and MD; D. Sundaram; Michael Gibbs; Helene Auriol Potier; Nitin Paranjpe; Chitra Nayak; Bobby Parikh; Govind Iyer; Americana Restaurants; GPT-4 Omni; Sunrise GmbH; Posti Group; Posti; Nordics; Hatch; Citizens Bank; Financial Capital; Natural Capital; Intellectual Capital; Social and Relationship Capital; Human Capital; Manufactured Capital; Live Enterprise; Infosys Topaz; Infosys Aster; Infosys Cobalt; Infosys

In [7]:
# Part 7 - Build Context

retrieved_docs = [
    doc
    for doc, score in docs_with_scores
]

context = "\n\n".join(
    doc.page_content[:800]
    for doc in retrieved_docs
)

In [8]:
# Part 8 - Create Prompt

prompt = f"""
You are a financial analyst.

Use ONLY the provided context.

Instructions:
- Answer using only the provided context.
- Use bullet points when appropriate.
- Mention financial figures exactly.
- Summarize instead of copying text.
- If information is unavailable, say:
"The information is not available in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

In [9]:
# Part 9 - Generate Answer

response = generator(
    prompt,
    max_new_tokens=150,
    do_sample=False
)

answer = response[0]["generated_text"]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
# Part 10 - Extract Final Answer

answer = answer.split(
    "Answer:"
)[-1].strip()

print("\nANSWER\n")

print(answer)


ANSWER

Infosys revenues were 1,36,592 crore for the year ended March 31, 2025.


In [11]:
# Part 11 - Show Sources

print("\nSOURCES\n")

for doc in retrieved_docs:

    print(
        f"{doc.metadata.get('source')} | "
        f"Page {doc.metadata.get('page_label')}"
    )


SOURCES

data\infosys_annual_repor.pdf | Page 30
data\infosys_annual_repor.pdf | Page 266
data\infosys_annual_repor.pdf | Page 257
